In [ ]:
# 加载环境变量
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage

load_dotenv()

# 提示词（Prompts）

发送给大模型的所有消息都可以称为**提示词（Prompt）**，它直接影响模型的输出结果。

## 1.系统提示词
在所有发送给LLM的消息中，System Message最为重要，它设定了模型的角色和聊天的背景。会影响到后续所有的对话。我们将其称之为**系统提示词（System Prompt）**。

在创建智能体时，就可以直接指定系统提示词。

In [ ]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

# 创建智能体
agent = create_agent(
    model = "deepseek-chat"
)

for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="月球的首都在哪里?")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)

In [ ]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

# 创建智能体
agent = create_agent(
    model = "deepseek-chat",
    system_prompt="你是一个科幻作家，根据用户的要求创建一个首都城市."
)

for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="月球的首都在哪里?")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)

# 2.Few-Shot examples

Few-shot示例是一种为模型提供多个示例的方法，以便它可以学习模式并生成更准确的响应。

In [ ]:

system_prompt = """
你是一个科幻作家，根据用户的要求创建一个太空之都。

用户：月球的首都是什么？
科幻作家：月华城（Lunara）

用户：火星的首都是什么？
科幻作家：赤晶城（Rubrum）

"""

# 创建智能体
agent = create_agent(
    model = "deepseek-chat",
    system_prompt=system_prompt
)

for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="金星的首都是什么?")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)


# 3.结构化输出
结构化输出允许你指定模型的输出格式，从而使模型的输出更易于解析和使用。



In [ ]:

system_prompt = """
你是一个科幻作家，根据用户的要求创建一个太空之都。请遵守以下结构，不要加markdown样式。

Name：首都的名字

Location：它所在的地方

Vibe：2-3个词来描述它的氛围

Economy：主要产业

"""

agent = create_agent(
    model="deepseek-chat",
    system_prompt=system_prompt
)

for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="月球的首都是什么?")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)

在LangChain中，实现结构化输出会更加简单。我们无需自己在提示词中添加描述实现结构化输出，而仅仅是设定好一个数据类型即可。


In [ ]:
from pydantic import BaseModel

# 首先，我们定义一个类，用来封装模型要输出的数据：
class CapitalInfo(BaseModel):
    name: str
    location: str
    vibe: str
    economy: str

In [ ]:
# 然后，我们就可以创建智能体并设置结构化输出的格式了。
agent = create_agent(
    model='deepseek-chat',
    system_prompt="你是一个科幻作家，根据用户的要求创建一个太空之都。",
    response_format=CapitalInfo # 设置结构化输出的格式
)

response = agent.invoke(
    {"messages": [HumanMessage(content="月球的首都是什么?")]}
)

In [ ]:
print(response)

In [ ]:
city = response['structured_response']
city

In [ ]:
print(f"{city.name}位于{city.location}，是一座{city.vibe}的城市，其主要产业包括{city.economy}。")